In [3]:
# Import required libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from api_dataset_loader import PlantDiseaseDataLoader

In [4]:
# Initialize the data loader
loader = PlantDiseaseDataLoader()

# Get all available classes
all_classes = loader.get_available_classes()
print(f"Total number of classes: {len(all_classes)}")
print("\nAvailable classes:")
for i, cls in enumerate(all_classes, 1):
    print(f"{i}. {cls}")

Total number of classes: 38

Available classes:
1. Apple___Apple_scab
2. Apple___Black_rot
3. Apple___Cedar_apple_rust
4. Apple___healthy
5. Blueberry___healthy
6. Cherry_(including_sour)___Powdery_mildew
7. Cherry_(including_sour)___healthy
8. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
9. Corn_(maize)___Common_rust_
10. Corn_(maize)___Northern_Leaf_Blight
11. Corn_(maize)___healthy
12. Grape___Black_rot
13. Grape___Esca_(Black_Measles)
14. Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
15. Grape___healthy
16. Orange___Haunglongbing_(Citrus_greening)
17. Peach___Bacterial_spot
18. Peach___healthy
19. Pepper,_bell___Bacterial_spot
20. Pepper,_bell___healthy
21. Potato___Early_blight
22. Potato___Late_blight
23. Potato___healthy
24. Raspberry___healthy
25. Soybean___healthy
26. Squash___Powdery_mildew
27. Strawberry___Leaf_scorch
28. Strawberry___healthy
29. Tomato___Bacterial_spot
30. Tomato___Early_blight
31. Tomato___Late_blight
32. Tomato___Leaf_Mold
33. Tomato___Septoria_leaf_sp

In [11]:
# Create a data generator that will load batches of images
def create_data_generator(classes, samples_per_class=50, batch_size=32):
    """Create a generator that yields batches of images"""
    while True:
        try:
            # Get a batch of images
            images, labels = loader.get_training_batch(classes, batch_size=batch_size)
            
            if images is None or labels is None or len(images) == 0 or len(labels) == 0:
                # Create dummy data if we couldn't load real data
                images = np.zeros((batch_size, 224, 224, 3))
                labels = np.zeros((batch_size, len(classes)))
                labels[:, 0] = 1  # Set first class as default
            
            # Convert to float32 and normalize
            images = images.astype('float32') / 255.0
            
            yield images, labels
        except Exception as e:
            print(f"Error in generator: {str(e)}")
            # Return dummy data on error
            images = np.zeros((batch_size, 224, 224, 3))
            labels = np.zeros((batch_size, len(classes)))
            labels[:, 0] = 1  # Set first class as default
            yield images, labels

# Create training and validation generators
train_gen = create_data_generator(all_classes, samples_per_class=50, batch_size=32)
valid_gen = create_data_generator(all_classes, samples_per_class=10, batch_size=32)

# Number of classes
num_classes = len(all_classes)
print(f"Training model for {num_classes} disease classes")

Training model for 38 disease classes


In [6]:
# Build the model
model = tf.keras.Sequential([
    # Input layer
    tf.keras.layers.Input(shape=(224, 224, 3)),
    
    # First Convolution Block
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Second Convolution Block
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Third Convolution Block
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Fourth Convolution Block
    tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(256, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Fifth Convolution Block
    tf.keras.layers.Conv2D(512, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(512, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Dropout for regularization
    tf.keras.layers.Dropout(0.25),
    
    # Flatten and Dense layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1500, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    
    # Output layer
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 222, 222, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 111, 111, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 54, 54, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 12, 12, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 5, 5, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 5, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1500)           │    19,201,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1500)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │        57,038 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,970,762 (91.44 MB)

 Trainable params: 23,970,762 (91.44 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Train the model
steps_per_epoch = 100  # Total training samples / batch_size
validation_steps = 20   # Total validation samples / batch_size

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=valid_gen,
    validation_steps=validation_steps,
    epochs=10
)

Error loading images for Pepper,_bell___Bacterial_spot: KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for Tomato___Leaf_Mold: KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for Raspberry___healthy: KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for Cherry_(including_sour)___healthy: KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for Grape___Esca_(Black_Measles): KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for Tomato___Septoria_leaf_spot: KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: KaggleApi.dataset_download_files() got an unexpected keyword argument 'target_dir'
Error loading images for S

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

# Plot training & validation accuracy values
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

# Save the trained model
model.save("trained_model_all_diseases.keras")
print("Model saved successfully!")

# Save training history
import json
with open("training_hist_all_diseases.json", "w") as f:
    json.dump(history.history, f)
print("Training history saved successfully!")

# Plant Disease Detection using Kaggle API
This notebook uses the Kaggle API to directly load plant disease images for training, working with all 38 disease classes.

In [2]:
```xml
<VSCode.Cell language="markdown">
# Plant Disease Detection using Kaggle API
This notebook uses the Kaggle API to directly load plant disease images for training, working with all 38 disease classes.
</VSCode.Cell>

<VSCode.Cell language="python">
# Import required libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from api_dataset_loader import PlantDiseaseDataLoader
</VSCode.Cell>

<VSCode.Cell language="python">
# Initialize the data loader
loader = PlantDiseaseDataLoader()

# Get all available classes
all_classes = loader.get_available_classes()
print(f"Total number of classes: {len(all_classes)}")
print("\nAvailable classes:")
for i, cls in enumerate(all_classes, 1):
    print(f"{i}. {cls}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Create a data generator that will load batches of images
def create_data_generator(classes, samples_per_class=50, batch_size=32):
    """Create a generator that yields batches of images"""
    while True:
        # Get a batch of images
        images, labels = loader.get_training_batch(classes, batch_size=batch_size)
        
        # Convert to float32 and normalize
        images = images.astype('float32') / 255.0
        
        yield images, labels
</VSCode.Cell>

<VSCode.Cell language="python">
# Create training and validation generators
train_gen = create_data_generator(all_classes, samples_per_class=50, batch_size=32)
valid_gen = create_data_generator(all_classes, samples_per_class=10, batch_size=32)

# Number of classes
num_classes = len(all_classes)
print(f"Training model for {num_classes} disease classes")
</VSCode.Cell>

<VSCode.Cell language="python">
# Build the model
model = tf.keras.Sequential([
    # Input layer
    tf.keras.layers.Input(shape=(224, 224, 3)),
    
    # First Convolution Block
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Second Convolution Block
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Third Convolution Block
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Fourth Convolution Block
    tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(256, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Fifth Convolution Block
    tf.keras.layers.Conv2D(512, 3, padding='same', activation='relu'),
    tf.keras.layers.Conv2D(512, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    
    # Dropout for regularization
    tf.keras.layers.Dropout(0.25),
    
    # Flatten and Dense layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1500, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    
    # Output layer
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()
</VSCode.Cell>

<VSCode.Cell language="python">
# Train the model
steps_per_epoch = 100  # Adjust based on your needs
validation_steps = 20   # Adjust based on your needs

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=valid_gen,
    validation_steps=validation_steps,
    epochs=10
)
</VSCode.Cell>

<VSCode.Cell language="python">
# Plot training history
plt.figure(figsize=(12, 4))

# Plot training & validation accuracy values
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()
</VSCode.Cell>

<VSCode.Cell language="python">
# Save the trained model
model.save("trained_model_all_diseases.keras")
print("Model saved successfully!")

# Save training history
import json
with open("training_hist_all_diseases.json", "w") as f:
    json.dump(history.history, f)
print("Training history saved successfully!")
</VSCode.Cell>

<VSCode.Cell language="python">
# Function to make predictions on new images
def predict_disease(image_path):
    """
    Make prediction on a single image
    """
    # Load and preprocess the image
    img = tf.keras.preprocessing.image.load_img(
        image_path, target_size=(224, 224)
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, 0)
    img_array /= 255.0
    
    # Make prediction
    prediction = model.predict(img_array)
    predicted_class = all_classes[np.argmax(prediction[0])]
    confidence = np.max(prediction[0])
    
    return predicted_class, confidence

# Example usage:
# class_name, confidence = predict_disease('path_to_image.jpg')
# print(f"Predicted class: {class_name}")
# print(f"Confidence: {confidence:.2%}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Generate classification report and confusion matrix
def evaluate_model(steps=20):
    """
    Evaluate the model and generate detailed metrics
    """
    # Collect predictions and true labels
    all_predictions = []
    all_true_labels = []
    
    for _ in range(steps):
        images, labels = next(valid_gen)
        predictions = model.predict(images)
        
        all_predictions.extend(np.argmax(predictions, axis=1))
        all_true_labels.extend(np.argmax(labels, axis=1))
    
    # Generate classification report
    from sklearn.metrics import classification_report
    print("\nClassification Report:")
    print(classification_report(all_true_labels, all_predictions, target_names=all_classes))
    
    # Generate confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(all_true_labels, all_predictions)
    
    # Plot confusion matrix
    plt.figure(figsize=(20, 20))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

# Run evaluation
evaluate_model()
</VSCode.Cell>
```

SyntaxError: invalid syntax (37351343.py, line 1)